# Compute a Gene Covariance Matrix from Atlas Streams

This tutorial shows how to compute a gene-by-gene covariance matrix from Atlas
expression minibatches in a single pass. It demonstrates how custom statistical
methods can process all selected cells without materializing the complete
cell-by-gene matrix in memory.

A streamed covariance matrix can support exploratory feature-correlation
analysis and small-scale dimensionality-reduction prototypes. For routine PCA,
use the built-in `sap.tl.pca()` workflow.

This tutorial builds on the single-pass streaming pattern introduced in
{doc}`stream-mean-and-variance`. Here, the same idea is extended from
per-gene summaries to a gene-by-gene covariance matrix, where memory planning
and accumulator updates become more important.

By the end of this tutorial, you will be able to:

- traverse an Atlas exactly once using dense minibatches;
- merge batch-level means and covariance accumulators;
- estimate a sample covariance matrix across all selected cells;
- plan memory use for gene-by-gene outputs;
- validate and save the resulting matrix.

## Before You Begin

This tutorial assumes that:

- quality control and preprocessing have been completed;
- an existing Atlas is open;
- the selected expression field is appropriate for covariance analysis;
- the selected number of genes is small enough for a dense
  gene-by-gene matrix.

The examples below assume an existing Atlas object:



In [1]:
import os
from pathlib import Path
import numpy as np
import scatlaspy as sap

os.chdir(Path("~/scAtlaspy-code-analysis").expanduser())

atlas_path = Path("./tmp/tutorials/basic_pbmc3k/pbmc3k_basic_copy.sasql")

if not atlas_path.is_file():
    raise FileNotFoundError(f"Atlas database not found: {atlas_path}")

atlas = sap.Atlas(
    atlas_path,
    db_memory_limit="8GB",
)



## 1. Define the Analysis View

Build a read index that defines the cells, genes, and expression representation
included in the calculation:



In [ ]:
atlas.build_read_index(
    cell_condition="filter_cells",
    gene_condition="filter_genes",
    use_hvg=True,
    use_data="data_count",
)

- count-scale expression values stored in `data_count`.

In [3]:
n_total = 0
mean = None
m2 = None



Traverse the selected cells once:



In [4]:
for batch_id, X_batch in enumerate(
    atlas.get_minibatch_dense(
        pass_mode="single-pass",
        batch_size=2048,
    ),
    start=1,
):
    X_batch = np.asarray(X_batch, dtype=np.float64)

    if X_batch.ndim != 2:
        raise ValueError("Each minibatch must be a two-dimensional matrix.")

    n_batch, n_features = X_batch.shape

    if n_batch == 0:
        continue

    if mean is not None and n_features != mean.shape[0]:
        raise ValueError(
            "The feature dimension changed between minibatches."
        )

    batch_mean = X_batch.mean(axis=0)
    centered = X_batch - batch_mean
    batch_m2 = centered.T @ centered

    if n_total == 0:
        n_total = n_batch
        mean = batch_mean.copy()
        m2 = batch_m2
        continue

    delta = batch_mean - mean
    new_total = n_total + n_batch

    # Add the within-batch covariance contribution.
    m2 += batch_m2

    # Reuse batch_m2 as workspace for the between-group correction.
    np.multiply.outer(delta, delta, out=batch_m2)
    batch_m2 *= n_total * n_batch / new_total
    m2 += batch_m2

    mean += delta * n_batch / new_total
    n_total = new_total

    if batch_id == 1 or batch_id % 100 == 0:
        print(f"Processed {n_total:,} cells")



This loop retains:

- one dense expression minibatch;
- its centered representation;
- the global mean vector;
- the gene-by-gene covariance accumulator.

It does not retain expression values from earlier minibatches.

```{note}
The calculation uses `float64` accumulation for numerical stability. Even when
the stored expression values use `float32`, covariance accumulation may benefit
from the additional precision.
```

## 5. Finalize the Covariance Matrix

The accumulator `m2` contains the summed cross-products around the global mean.

Calculate the sample covariance matrix using \(n-1\) in the denominator:



In [5]:
if n_total < 2:
    raise ValueError(
        "At least two cells are required to calculate covariance."
    )

m2 /= n_total - 1
covariance = m2

print(f"Processed {n_total:,} cells")
print(f"Mean vector shape: {mean.shape}")
print(f"Covariance matrix shape: {covariance.shape}")


Processed 2,700 cells
Mean vector shape: (2000,)
Covariance matrix shape: (2000, 2000)



This operation reuses the accumulator array rather than allocating another
complete gene-by-gene matrix.

For a population covariance estimate, divide by `n_total` instead. Most
statistical workflows use the sample covariance definition shown above.

## 6. Validate the Result

Check that the output has the expected dimensions and contains finite values:



In [6]:
if covariance.ndim != 2:
    raise ValueError("The covariance output is not a matrix.")

if covariance.shape[0] != covariance.shape[1]:
    raise ValueError("The covariance matrix is not square.")

if covariance.shape[0] != mean.shape[0]:
    raise ValueError(
        "The mean vector and covariance matrix use different feature counts."
    )

if not np.isfinite(mean).all():
    raise ValueError("The streamed mean contains non-finite values.")

if not np.isfinite(covariance).all():
    raise ValueError("The covariance matrix contains non-finite values.")



Check numerical symmetry:



In [7]:
max_asymmetry = np.max(np.abs(covariance - covariance.T))

print(f"Maximum asymmetry: {max_asymmetry:.3e}")

if not np.allclose(
    covariance,
    covariance.T,
    rtol=1e-7,
    atol=1e-8,
):
    raise ValueError("The covariance matrix is not numerically symmetric.")


Maximum asymmetry: 0.000e+00



Inspect the variance range:



In [8]:
variances = np.diag(covariance)

print(f"Minimum variance: {variances.min():.6f}")
print(f"Maximum variance: {variances.max():.6f}")


Minimum variance: 0.098302
Maximum variance: 1.000371


```{note}
When `use_data="data_count"`, the covariance diagonal reflects per-gene count
variance, which depends on expression magnitude. For many analyses,
log-normalized or scaled expression may be more interpretable.
```

In [9]:
X_reference = np.vstack([
    np.asarray(batch, dtype=np.float64)
    for batch in atlas.get_minibatch_dense(
        pass_mode="single-pass",
        batch_size=2048,
    )
])

reference_covariance = np.cov(
    X_reference,
    rowvar=False,
    ddof=1,
)



Compare the streamed and in-memory results only when both calculations use:

- exactly the same cells;
- exactly the same genes;
- exactly the same gene order;
- the same expression representation;
- the same covariance denominator.



In [10]:
np.testing.assert_allclose(
    covariance,
    reference_covariance,
    rtol=1e-6,
    atol=1e-8,
)



A small reference calculation is one of the most effective ways to test a
custom streaming statistic before applying it to a large Atlas.

## 8. Use the Covariance Matrix

For a moderate number of genes, calculate eigenvalues and eigenvectors with
NumPy:



In [11]:
eigvals, eigvecs = np.linalg.eigh(covariance)

order = np.argsort(eigvals)[::-1]
eigvals = eigvals[order]
eigvecs = eigvecs[:, order]



Because covariance matrices are symmetric, `np.linalg.eigh()` is more
appropriate than the general `np.linalg.eig()` function.

Inspect the leading eigenvalues:



In [12]:
print(eigvals[:10])


[40.31335415 20.17302629 15.68850815 14.02031106  8.67319884  4.67047989
  4.07464487  3.31250962  3.26203942  3.19030311]



```{warning}
A complete eigendecomposition has approximately \(O(p^3)\) computational cost
and returns a dense \(p \times p\) eigenvector matrix.

For large feature sets, use truncated, randomized, incremental, or
matrix-free methods rather than calculating every eigenvector of the full
covariance matrix.
```

For routine atlas-scale dimensionality reduction, use the built-in PCA
workflow in the main analysis pipeline rather than recomputing it here:



In [13]:
# sap.tl.pca(
#     atlas,
#     n_components=50,
# )



## 9. Save the Result

Save the mean vector and covariance matrix:



In [14]:
from pathlib import Path

output_path = Path("./results/streamed_covariance.npz")
output_path.parent.mkdir(parents=True, exist_ok=True)

np.savez(
    output_path,
    n_cells=n_total,
    mean=mean,
    covariance=covariance,
)

print(f"Saved covariance result to {output_path}")


Saved covariance result to results/streamed_covariance.npz



Also save or record:

- the Atlas used for the calculation;
- the selected cells;
- the selected genes and their exact order;
- the expression field;
- the filtering and HVG settings;
- the covariance denominator;
- the batch size and software version.

Without the associated gene order, the rows and columns of the covariance
matrix cannot be mapped reliably back to biological features.

## Limitations

This approach is useful when the number of cells is large but the selected
feature set is moderate.

It does not eliminate:

- quadratic storage in the number of genes;
- quadratic computation per cell;
- the cost of dense expression minibatches;
- the cubic cost of a complete eigendecomposition.

For very large feature sets, consider:

- reducing the feature set;
- computing selected gene-gene relationships only;
- randomized or incremental dimensionality reduction;
- low-rank sketches;
- sparse or block-structured approximations;
- matrix-free optimization methods.



## Close the Atlas

Close the database connection when this tutorial is complete. This releases
the DuckDB file lock so the same `.sasql` Atlas can be opened by another
notebook or Python session.


In [15]:
atlas.close()


## Next Steps

See {doc}`implement-minibatch-kmeans` for an iterative method that uses
randomized, multi-pass minibatches rather than a deterministic single-pass
summary.

See {doc}`stream-mean-and-variance` for a linear-memory streaming statistic that
does not require a gene-by-gene output matrix.